In [1]:
!nvidia-smi

Tue Jul 28 22:48:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'

In [18]:
import os
print(os.listdir(dataset_path))

['labels', 'images']


In [6]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: Tesla T4


In [7]:
# 1. Install dependencies
!pip install -q --upgrade ultralytics albumentations opencv-python-headless tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 82.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 25.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
tensorflow 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.21.0 which is incompatible.
google-clo

In [52]:
import os
base_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'
print("Folders found:", os.listdir(base_path))

Folders found: ['labels', 'images']


In [41]:
!pip install -q ultralytics

In [42]:
print('\n--- Verifying ultralytics installation ---')
!pip show ultralytics
print('------------------------------------------')


--- Verifying ultralytics installation ---
Name: ultralytics
Version: 8.4.109
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-thop
Required-by: 
------------------------------------------


In [57]:
import os
import yaml

# Exact path printed by your output
dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'

# Verify subdirectories directly
subfolders = os.listdir(dataset_path) if os.path.exists(dataset_path) else []
print("Directories found inside dataset:", subfolders)

# Auto-detect train/val/test subfolder naming conventions
train_folder = 'train/images' if 'train' in subfolders else 'images/train'
val_folder = 'val/images' if 'val' in subfolders else ('valid/images' if 'valid' in subfolders else 'images/val')
test_folder = 'test/images' if 'test' in subfolders else ('images/test' if 'images' in subfolders else val_folder)

yaml_content = {
    'path': dataset_path,
    'train': train_folder,
    'val': val_folder,
    'test': test_folder,
    'nc': 33,
    'names': [
        '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k',
        'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u',
        'v', 'w', 'x'
    ]
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print("data.yaml updated successfully!")

Directories found inside dataset: ['labels', 'images']
data.yaml updated successfully!


In [58]:
from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path
import shutil
import yaml
import subprocess

_ultralytics_available = False
try:
    from ultralytics import YOLO
    _ultralytics_available = True
except ImportError:
    print("\n[CRITICAL ERROR] 'ultralytics' is not installed in the active environment. Please run: !pip install ultralytics\n")

KAGGLE_WORKING_DIR = Path("/kaggle/working")
DEFAULT_DATA_YAML = KAGGLE_WORKING_DIR / "data.yaml"
DEFAULT_MODEL = "yolo11n.pt"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train YOLO11n model on Kaggle dataset with evaluation artifacts.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--data",
        "-d",
        type=Path,
        default=DEFAULT_DATA_YAML,
        help="Path to YOLO dataset configuration file (data.yaml).",
    )
    parser.add_argument(
        "--model",
        "-m",
        type=str,
        default=DEFAULT_MODEL,
        help="Starting model weights or architecture (e.g., yolo11n.pt).",
    )
    parser.add_argument(
        "--epochs",
        "-e",
        type=int,
        default=50,
        help="Number of training epochs.",
    )
    parser.add_argument(
        "--imgsz",
        type=int,
        default=640,
        help="Training image resolution.",
    )
    parser.add_argument(
        "--batch",
        "-b",
        type=int,
        default=-1,
        help="Training batch size (-1 for AutoBatch).",
    )
    parser.add_argument(
        "--project",
        type=Path,
        default=KAGGLE_WORKING_DIR / "runs" / "detect",
        help="Ultralytics project output directory.",
    )
    parser.add_argument(
        "--name",
        type=str,
        default="train_yolo11n",
        help="Ultralytics run name for output artifacts.",
    )
    parser.add_argument(
        "--device",
        default=0,
        help="Target device ('0' for CUDA GPU, 'cpu', or leave empty for auto-detect).",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for reproducible training.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=8,
        help="Number of dataloader worker threads.",
    )
    parser.add_argument(
        "--cache",
        action="store_true",
        default=False,
        help="Cache images in RAM.",
    )
    parser.add_argument(
        "--no-amp",
        action="store_true",
        help="Disable Automatic Mixed Precision (AMP).",
    )
    return parser.parse_known_args()[0]


def format_time(seconds: float) -> str:
    """Format seconds into HHh MMm SSs."""
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes:02d}m {secs:02d}s"


def verify_thesis_artifacts(run_dir: Path) -> list[tuple[str, bool]]:
    """Check for existence of evaluation plots & metrics."""
    expected_artifacts = [
        ("weights/best.pt (Top Model Checkpoint)", run_dir / "weights" / "best.pt"),
        ("weights/last.pt (Final Epoch Checkpoint)", run_dir / "weights" / "last.pt"),
        ("F1_curve.png (F1 Score vs Confidence)", run_dir / "F1_curve.png"),
        ("PR_curve.png (Precision-Recall Curve)", run_dir / "PR_curve.png"),
        ("P_curve.png (Precision vs Confidence)", run_dir / "P_curve.png"),
        ("R_curve.png (Recall vs Confidence)", run_dir / "R_curve.png"),
        ("confusion_matrix.png (Confusion Matrix)", run_dir / "confusion_matrix.png"),
        ("confusion_matrix_normalized.png (Normalized Confusion Matrix)", run_dir / "confusion_matrix_normalized.png"),
        ("results.png (Loss & Metric Epoch Curves)", run_dir / "results.png"),
        ("results.csv (Numerical Loss/Metric Log)", run_dir / "results.csv"),
    ]

    return [(name, path.exists()) for name, path in expected_artifacts]


def resolve_kaggle_data_yaml(input_arg_path: Path) -> Path:
    """Check /kaggle/working/ for data.yaml or dataset.yaml."""
    candidate_paths = [
        input_arg_path,
        KAGGLE_WORKING_DIR / "data.yaml",
        KAGGLE_WORKING_DIR / "dataset.yaml",
        Path.cwd() / "data.yaml",
        Path.cwd() / "dataset.yaml",
    ]

    for candidate in candidate_paths:
        if candidate.exists():
            print(f"[INFO] Located configuration file at: {candidate.resolve()}")
            return candidate.resolve()

    raise FileNotFoundError(
        "[ERROR] Could not find 'data.yaml' or 'dataset.yaml' in /kaggle/working/.\n"
        "Please run Cell 1 to generate data.yaml."
    )


def run_yolo_training(args: argparse.Namespace) -> None:
    data_path = resolve_kaggle_data_yaml(args.data)
    run_dir = args.project.resolve() / args.name

    print("=" * 80)
    print(" YOLO11 Thesis Model Trainer & Evaluator (Kaggle Environment) ")
    print("=" * 80)
    print(f" Model Architecture     : {args.model}")
    print(f" Dataset Configuration  : {data_path}")
    print(f" Training Epochs        : {args.epochs}")
    print(f" Image Resolution       : {args.imgsz}x{args.imgsz}")
    print(f" Batch Size             : {args.batch}")
    print(f" Run Directory          : {run_dir}")
    print(f" Random Seed            : {args.seed}")
    print("-" * 80)

    print(f"[INFO] Initializing YOLO model '{args.model}'...")
    model = YOLO(args.model)

    start_time = time.time()

    print("\n[INFO] Starting model training...")
    results = model.train(
        data=str(data_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        project=str(args.project.resolve()),
        name=args.name,
        exist_ok=True,
        plots=True,
        save=True,
        seed=args.seed,
        workers=args.workers,
        device=args.device,
        cache=args.cache,
        amp=not args.no_amp,
    )

    elapsed_training = time.time() - start_time
    print(f"\n[SUCCESS] Model training completed in {format_time(elapsed_training)}!")

    best_weights_path = run_dir / "weights" / "best.pt"
    eval_model = YOLO(str(best_weights_path)) if best_weights_path.exists() else model

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: VALIDATION SET PERFORMANCE ")
    print("=" * 80)
    val_metrics = eval_model.val(data=str(data_path), split="val", imgsz=args.imgsz, plots=True)

    print("\n" + "=" * 80)
    print(" THESIS EVALUATION: TEST SET FINAL BENCHMARK ")
    print("=" * 80)
    test_metrics = eval_model.val(data=str(data_path), split="test", imgsz=args.imgsz, plots=True)

    print("\n" + "=" * 80)
    print(" MASTER'S / PHD THESIS PERFORMANCE SUMMARY ")
    print("=" * 80)
    
    header = f"{'Split':<12} | {'Precision (P)':<15} | {'Recall (R)':<15} | {'mAP@50':<15} | {'mAP@50-95':<15}"
    print(header)
    print("-" * 80)

    val_p = val_metrics.results_dict.get("metrics/precision(B)", 0.0)
    val_r = val_metrics.results_dict.get("metrics/recall(B)", 0.0)
    val_map50 = val_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    val_map = val_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    test_p = test_metrics.results_dict.get("metrics/precision(B)", 0.0)
    test_r = test_metrics.results_dict.get("metrics/recall(B)", 0.0)
    test_map50 = test_metrics.results_dict.get("metrics/mAP50(B)", 0.0)
    test_map = test_metrics.results_dict.get("metrics/mAP50-95(B)", 0.0)

    val_row = f"{'Validation':<12} | {val_p:<15.4f} | {val_r:<15.4f} | {val_map50:<15.4f} | {val_map:<15.4f}"
    test_row = f"{'Test Set':<12} | {test_p:<15.4f} | {test_r:<15.4f} | {test_map50:<15.4f} | {test_map:<15.4f}"

    print(val_row)
    print(test_row)
    print("=" * 80)

    print("\n" + "=" * 80)
    print(" GENERATED THESIS FIGURES & ARTIFACTS VERIFICATION ")
    print("=" * 80)
    artifacts = verify_thesis_artifacts(run_dir)
    for artifact_name, status in artifacts:
        symbol = "[✓]" if status else "[×]"
        print(f" {symbol} {artifact_name}")
    print("=" * 80 + "\n")

In [59]:
# Create training parameters pointing directly to your working data.yaml
args = parse_args()

# Start the training and evaluation pipeline
run_yolo_training(args)

[INFO] Located configuration file at: /kaggle/working/data.yaml
 YOLO11 Thesis Model Trainer & Evaluator (Kaggle Environment) 
 Model Architecture     : yolo11n.pt
 Dataset Configuration  : /kaggle/working/data.yaml
 Training Epochs        : 50
 Image Resolution       : 640x640
 Batch Size             : -1
 Run Directory          : /kaggle/working/runs/detect/train_yolo11n
 Random Seed            : 42
--------------------------------------------------------------------------------
[INFO] Initializing YOLO model 'yolo11n.pt'...

[INFO] Starting model training...
Ultralytics 8.4.109 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working

In [65]:
import os
import yaml

# Corrected class list (32 classes: 1-9 and letters A-X excluding J and Z)
correct_names = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k',
    'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u',
    'v', 'w', 'x'
]

dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'

# Check actual subfolders present in the dataset path
subfolders = os.listdir(dataset_path) if os.path.exists(dataset_path) else []

# Dynamically set subfolder paths based on directory layout
train_folder = 'train/images' if 'train' in subfolders else 'images/train'
val_folder = 'val/images' if 'val' in subfolders else ('valid/images' if 'valid' in subfolders else 'images/val')
test_folder = 'test/images' if 'test' in subfolders else ('images/test' if 'images' in subfolders else val_folder)

yaml_content = {
    'path': dataset_path,
    'train': train_folder,
    'val': val_folder,
    'test': test_folder,
    'nc': 32,
    'names': correct_names
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print(f"data.yaml updated successfully using validation path: '{val_folder}'")

data.yaml updated successfully using validation path: 'images/val'


In [66]:
from ultralytics import YOLO

best_weights_path = '/kaggle/working/runs/detect/train_yolo11n/weights/best.pt'
model = YOLO(best_weights_path)

metrics = model.val(
    data='/kaggle/working/data.yaml',
    split='val',
    imgsz=640,
    plots=True
)

print("\n--- RE-EVALUATION COMPLETE ---")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.109 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,588,587 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 358.7±182.7 MB/s, size: 270.5 KB)
val: Scanning /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/labels/val... 5037 images, 0 backgrounds, 156 corrupt: 100% ━━━━━━━━━━━━ 5037/5037 1.1Kit/s 4.8s0.0s
val: /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/images/val/y_20260724_144219_1b1b1c_000001_aug3.jpg: ignoring corrupt image/label: Label class 32 exceeds dataset class count 32. Possible class labels are 0-31
val: /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/images/val/y_20260724_144219_1b1b1c_000002_aug2.jpg: ignoring corrupt image/label: Label class 32 exceeds dataset class count 32. Possible class labels are 0-31
val: /kaggle/input/datasets/adelekeadeniyan/

In [67]:
import os
import yaml

# Exact 32 class names mapped to indices 0-31
correct_32_classes = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k',
    'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u',
    'v', 'w', 'x'  # Or 'y' depending on your final letter class
]

dataset_path = '/kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset'

subfolders = os.listdir(dataset_path) if os.path.exists(dataset_path) else []
val_folder = 'val/images' if 'val' in subfolders else ('valid/images' if 'valid' in subfolders else 'images/val')
train_folder = 'train/images' if 'train' in subfolders else 'images/train'
test_folder = 'test/images' if 'test' in subfolders else ('images/test' if 'images' in subfolders else val_folder)

yaml_content = {
    'path': dataset_path,
    'train': train_folder,
    'val': val_folder,
    'test': test_folder,
    'nc': 32,
    'names': correct_32_classes
}

with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(yaml_content, f)

print(f"data.yaml updated to {len(correct_32_classes)} classes starting from '1'.")

data.yaml updated to 32 classes starting from '1'.


In [68]:
from ultralytics import YOLO

# Load best trained weights
best_weights_path = '/kaggle/working/runs/detect/train_yolo11n/weights/best.pt'
model = YOLO(best_weights_path)

# Run validation against updated data.yaml
metrics = model.val(
    data='/kaggle/working/data.yaml',
    split='val',
    imgsz=640,
    plots=True
)

print("\n--- EVALUATION DISPLAY UPDATED ---")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.109 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,588,587 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 589.2±673.0 MB/s, size: 604.7 KB)
val: Scanning /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/labels/val... 5037 images, 0 backgrounds, 156 corrupt: 100% ━━━━━━━━━━━━ 5037/5037 1.0Kit/s 4.8s.1s
val: /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/images/val/y_20260724_144219_1b1b1c_000001_aug3.jpg: ignoring corrupt image/label: Label class 32 exceeds dataset class count 32. Possible class labels are 0-31
val: /kaggle/input/datasets/adelekeadeniyan/nsl-alphanumeric-dataset/content/split_dataset/images/val/y_20260724_144219_1b1b1c_000002_aug2.jpg: ignoring corrupt image/label: Label class 32 exceeds dataset class count 32. Possible class labels are 0-31
val: /kaggle/input/datasets/adelekeadeniyan/n

In [70]:
import torch

# Path to your top checkpoint
weights_path = '/kaggle/working/runs/detect/train_yolo11n/weights/best.pt'

# Load the PyTorch checkpoint allowing object unpickling
checkpoint = torch.load(weights_path, map_location='cpu', weights_only=False)

# Define your exact 32 real-world class names ending in 'y'
correct_32_names = {
    0: '1', 1: '2', 2: '3', 3: '4', 4: '5', 5: '6', 6: '7', 7: '8', 8: '9',
    9: 'a', 10: 'b', 11: 'c', 12: 'd', 13: 'e', 14: 'f', 15: 'g', 16: 'h', 17: 'i',
    18: 'k', 19: 'l', 20: 'm', 21: 'n', 22: 'o', 23: 'p', 24: 'q', 25: 'r',
    26: 's', 27: 't', 28: 'u', 29: 'v', 30: 'w', 31: 'y'
}

# Update internal names dictionary on the model object
if 'model' in checkpoint and hasattr(checkpoint['model'], 'names'):
    checkpoint['model'].names = correct_32_names

if 'ema' in checkpoint and hasattr(checkpoint['ema'], 'names'):
    checkpoint['ema'].names = correct_32_names

# Save the updated weights back to disk
torch.save(checkpoint, weights_path)

print("Checkpoint successfully updated! Class index 31 is now mapped to 'y'.")

Checkpoint successfully updated! Class index 31 is now mapped to 'y'.


In [77]:
from ultralytics import YOLO

# Load updated model
model = YOLO('/kaggle/working/runs/detect/train_yolo11n/weights/best.pt')

# Confirm that index 31 maps to 'y'
print("Last class in model metadata:", model.names[31])

Last class in model metadata: y


In [75]:
import os
import shutil

folder_to_zip = '/kaggle/working/runs'
output_filename = '/kaggle/working/runs'
zip_file_path = f"{output_filename}.zip"

# 1. Check if the previous zip file exists and remove it
if os.path.exists(zip_file_path):
    os.remove(zip_file_path)
    print(f"Removed old archive: {zip_file_path}")

# 2. Verify the source runs folder exists and create the new zip
if os.path.exists(folder_to_zip):
    print(f"Zipping folder: {folder_to_zip}...")
    shutil.make_archive(output_filename, 'zip', folder_to_zip)
    print("Successfully created updated runs.zip archive!")
else:
    print(f"Error: Folder '{folder_to_zip}' not found.")

Removed old archive: /kaggle/working/runs.zip
Zipping folder: /kaggle/working/runs...
Successfully created updated runs.zip archive!


In [79]:
import cv2
from ultralytics import YOLO

# Provide the exact full path to your saved weights
model_path = '/kaggle/working/runs/detect/train_yolo11n/weights/best.pt'
model = YOLO(model_path)

# Verify that class 31 displays 'y'
print("Verified class 31:", model.names[31])

# Launch webcam stream
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, stream=True)

    for r in results:
        annotated_frame = r.plot()

    cv2.imshow("NSL Real-Time Alphanumeric Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Verified class 31: y


[ WARN:0@22160.236] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@22160.237] global cap_ffmpeg_impl.hpp:1246 open VIDEOIO/FFMPEG: OpenCV should be configured with libavdevice to open a camera device


error: OpenCV(5.0.0) /io/opencv/modules/highgui/src/window.cpp:1210: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'destroyAllWindowsImpl'


In [1]:
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js if 'google.colab' in str(get_ipython()) else (None, None)
import cv2
import numpy as np
from ultralytics import YOLO
import base64

# Load updated YOLO model
model = YOLO('/kaggle/working/runs/detect/train_yolo11n/weights/best.pt')

# JavaScript for capturing a single frame from the browser webcam inside Kaggle UI
js_code = """
async function captureWebcam() {
    const div = document.createElement('div');
    const video = document.createElement('video');
    const button = document.createElement('button');
    
    button.textContent = '📸 Capture Hand Sign';
    button.style.padding = '10px 15px';
    button.style.margin = '10px 0';
    button.style.cursor = 'pointer';
    
    div.appendChild(video);
    div.appendChild(button);
    document.body.appendChild(div);
    
    const stream = await navigator.mediaDevices.getUserMedia({ video: true });
    video.srcObject = stream;
    await video.play();
    
    await new Promise((resolve) => button.onclick = resolve);
    
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    
    stream.getTracks().forEach(track => track.stop());
    div.remove();
    return canvas.toDataURL('image/jpeg', 0.85);
}
captureWebcam();
"""

print("Click '📸 Capture Hand Sign' in the frame below to run live detection:")

# Trigger JS camera interface
from IPython.display import HTML
display(HTML('''
<script>
''' + js_code + '''
</script>
'''))

SyntaxError: invalid syntax (3650360668.py, line 2)